# 🚨 Notebook 3 — Writing Compensations That Actually Work

Writing the happy path is easy. Writing **good compensations** is where sagas succeed or fail
in production. This notebook walks through four real-world problems and how to handle them.

| # | Problem | Technique |
|---|---------|-----------|
| 1 | Compensation may run twice after a retry | **Idempotency** |
| 2 | A step fails *temporarily* — don't compensate yet! | **Retry before compensating** |
| 3 | You can't literally un-send an email | **Semantic compensation** |
| 4 | After some step, going back is impossible | **Pivot transaction / forward-only recovery** |


## 🛠️ Setup

```bash
cd 05-microservices/saga
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Idempotent compensations

A *compensation* can be triggered multiple times:
- the orchestrator crashes after calling `refund()` but before recording it,
- an event is delivered twice by your message broker (at-least-once delivery is the norm).

👉 Your compensation **must produce the same end-state no matter how many times it runs**.

Below, `refund_naive` doubles the refund on the second call — that's a bug.
`refund_idempotent` uses an "already-refunded" set keyed by `order_id`, so running it twice is safe.


In [ ]:
payments = {"refunds": 0}
refunded_ids = set()

def refund_naive(order_id, amount):
    payments["refunds"] += amount  # 💥 every call adds again
    print(f"  [naive]  refunds = {payments['refunds']}")

def refund_idempotent(order_id, amount):
    if order_id in refunded_ids:
        print(f"  [idem]   order {order_id} already refunded — skip")
        return
    payments["refunds"] += amount
    refunded_ids.add(order_id)
    print(f"  [idem]   refunds = {payments['refunds']}")

print("— naive refund called twice —")
refund_naive(order_id=1, amount=20)
refund_naive(order_id=1, amount=20)
print("total refunds:", payments["refunds"], "(should be 20, bug shows 40)")

print("\n— idempotent refund called twice —")
payments["refunds"] = 0
refund_idempotent(order_id=2, amount=20)
refund_idempotent(order_id=2, amount=20)
print("total refunds:", payments["refunds"], "(correct: 20)")


**Takeaway:** Give every saga instance a stable **saga id** (or order id). Every compensation checks:
*"have I already done this for this id?"*. In SQL terms: a `UNIQUE` constraint on
`(saga_id, step_name)` in a `compensations` table is usually enough.


## 2. Retry *before* you compensate

Most production failures are **transient**: a brief network hiccup, a slow database, a 5xx from a dependency.
Compensating on the first error means you'll cancel perfectly good orders because the courier API blinked.

The rule of thumb:

1. If a step fails, **retry** a few times (with exponential backoff + jitter).
2. Only after **N retries** or a **non-retryable error** (400, "insufficient funds", …) do you compensate.


In [ ]:
import time, random

class Flaky:
    """Succeeds after `fail_times` failures — simulates a flaky dependency."""
    def __init__(self, fail_times=2): self.left = fail_times
    def __call__(self):
        if self.left > 0:
            self.left -= 1
            raise RuntimeError("transient: connection reset")
        return "ok"

def with_retry(fn, tries=5, base=0.05):
    for attempt in range(1, tries + 1):
        try:
            return fn()
        except RuntimeError as e:
            if attempt == tries:
                # Don't sleep after the final attempt — nobody is waiting on it.
                print(f"  attempt {attempt} failed ({e}); out of attempts")
                break
            sleep = base * (2 ** (attempt - 1)) + random.random() * 0.01
            print(f"  attempt {attempt} failed ({e}); retry in {sleep:.2f}s")
            time.sleep(sleep)
    raise RuntimeError(f"gave up after {tries} attempts")

print("→ call flaky dependency with retry")
print("result:", with_retry(Flaky(fail_times=2)))


### Retryable vs non-retryable — make the distinction *executable*

Prose about "split errors into two buckets" is easy to nod along to and easy to get
wrong in code. The bug it prevents is expensive in both directions:

- Retrying a **non-retryable** error ("card declined") just makes the customer wait
  5× longer for the same "no", while the saga holds inventory hostage.
- Compensating on a **retryable** error cancels perfectly good orders because a
  network card blinked.

So classify explicitly, and let the saga runner branch on the answer:

In [ ]:
class Transient(Exception):
    """Network blip, 5xx, timeout, deadlock — try again."""

class Terminal(Exception):
    """Card declined, validation error, 4xx — trying again changes nothing."""

def classify(exc):
    return 'retry' if isinstance(exc, Transient) else 'compensate'

def run_step(name, fn, tries=3, base=0.01):
    """Retry transient failures; surface terminal ones immediately."""
    for attempt in range(1, tries + 1):
        try:
            return fn()
        except Exception as e:
            if classify(e) == 'compensate':
                print(f"  ✗ {name}: {e} → TERMINAL, compensate now "
                      f"(after {attempt} attempt{'s' if attempt > 1 else ''})")
                raise
            if attempt == tries:
                print(f"  ✗ {name}: {e} → transient but out of retries, compensate")
                raise
            print(f"  … {name}: {e} → transient, retry {attempt}/{tries - 1}")
            time.sleep(base * (2 ** (attempt - 1)))

# --- Case A: a transient failure that recovers. We must NOT compensate. ---
blips = {'n': 0}
def flaky_charge():
    blips['n'] += 1
    if blips['n'] < 3:
        raise Transient("connection reset by peer")
    return "charged"

print("Case A — transient failure:")
print("  result:", run_step("charge", flaky_charge), "→ no compensation needed ✅\n")

# --- Case B: a terminal failure. Retrying is pure waste. ---
attempts = {'n': 0}
def declined_charge():
    attempts['n'] += 1
    raise Terminal("card declined: insufficient funds")

print("Case B — terminal failure:")
try:
    run_step("charge", declined_charge)
except Terminal:
    pass
print(f"  the card was charged-attempted {attempts['n']} time(s) — not 3. ✅")
print("  A naive `except Exception: retry` would have made the customer wait")
print("  for three identical declines before showing the error.")

## 3. Semantic compensation: the un-sendable email

You cannot "un-send" an email, SMS, push notification, or physical shipment.
But you can send a **new** message that cancels the business effect. That's **semantic compensation**.

| Step                             | Naive compensation    | Semantic compensation                               |
|----------------------------------|------------------------|------------------------------------------------------|
| `send_order_confirmation_email`  | "delete email" (🙃)    | `send_order_cancelled_email` with apology           |
| `print_shipping_label`           | "un-print"             | `send_return_label`                                  |
| `ship_package`                   | "un-ship"              | `request_return_pickup`                              |


In [ ]:
sent_emails = []

def send_confirmation(order_id, customer):
    msg = f"Hi {customer}, your order #{order_id} is confirmed."
    sent_emails.append(msg); print("  ✉️", msg)

def send_apology(order_id, customer):
    # "compensation" = a new outbound email that apologises and explains
    msg = (f"Hi {customer}, we're sorry — order #{order_id} could not be completed. "
           "You have not been charged.")
    sent_emails.append(msg); print("  ✉️", msg)

send_confirmation(42, "Ada")
# ... later, saga fails ...
send_apology(42, "Ada")
print("\nmailbox:")
for m in sent_emails: print(" -", m)


## 4. Pivot transaction — the point of no return

In most sagas there's a step after which you simply **can't go back**.
Classic examples:

- the warehouse forklifts have already loaded the container onto a truck,
- you've already transferred cash to an external bank,
- a physical piece of mail has been dropped in the post box.

This step is called the **pivot transaction**. The rule is:

- **Before** the pivot: failures → **compensate** (backward recovery).
- **At or after** the pivot: failures → **retry forever / escalate to humans** (forward recovery).

Below, `ship` is the pivot. If `notify` fails *after* shipping, we don't un-ship —
we retry the notification until it succeeds, or we open a support ticket.


In [ ]:
from dataclasses import dataclass
from typing import Callable, Optional

@dataclass
class Step:
    name: str
    do: Callable[[], None]
    undo: Optional[Callable[[], None]] = None   # None = pivot or later (no backward recovery)

class SagaWithPivot:
    def __init__(self, steps): self.steps = steps
    def run(self):
        done = []
        for i, s in enumerate(self.steps):
            try:
                print(f"→ {s.name}")
                s.do()
                done.append(s)
            except Exception as e:
                print(f"✗ {s.name} failed: {e}")
                if s.undo is None:
                    print("  ⚠️ past the pivot — cannot compensate; retry or escalate.")
                    return "needs_human"
                for d in reversed(done):
                    if d.undo is None:
                        print(f"  ⚠️ cannot compensate '{d.name}' (pivot) — escalating")
                        return "needs_human"
                    print(f"  ↶ undo {d.name}")
                    d.undo()
                return "compensated"
        return "committed"

state = {"notified": False}
def reserve(): pass
def release(): pass
def charge():  pass
def refund():  pass
def ship():    pass                       # PIVOT: no undo
def notify():  raise RuntimeError("email provider down")

result = SagaWithPivot([
    Step("reserve", reserve, release),
    Step("charge",  charge,  refund),
    Step("ship",    ship,    undo=None),  # 🚩 pivot
    Step("notify",  notify,  undo=None),  # after pivot — forward-only
]).run()
print("\nresult:", result)


## ✅ Summary — a checklist for writing compensations

- [ ] Every compensation is **idempotent** (safe to run twice).
- [ ] A failing compensation **does not abort the remaining ones** — each `undo` gets
      its own `try`, its own retries, and a dead-letter entry if it stays broken
      (see notebook 1).
- [ ] Retryable vs non-retryable errors are **distinguished**; retry transient ones.
- [ ] Compensations are **forward business actions** (new entries in the ledger), not magic "undo".
- [ ] Where real-world side-effects cannot be reversed (email, SMS, shipped box), use a
      **semantic compensation**.
- [ ] Identify your **pivot transaction**; after it, move to forward-only recovery and alert humans.
- [ ] Persist saga state to a **saga log** so you can resume after a crash (next lab / extension).

### Real-world systems that implement these ideas
- **Temporal** / **Cadence** — orchestration + durable state + automatic retries.
- **AWS Step Functions** — visual orchestration with built-in error catchers & retries.
- **Kafka + transactional outbox** — choreography style with reliable event delivery.
- **Netflix Conductor**, **Uber Cadence**, **Camunda** — production saga engines.
